# EDA — NMRShiftDB2 Raw Dataset

In [ ]:
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

SDF_PATH = "data/raw/nmrshiftdb2withsignals.sd"
SHIFT_MIN, SHIFT_MAX = -50.0, 300.0

In [ ]:
supplier = Chem.SDMolSupplier(SDF_PATH, removeHs=True, sanitize=True)

n_total = 0
n_parsed = 0
n_with_13c = 0

atom_presence      = Counter()  
heavy_atom_counts  = []       
mol_weights        = []        
all_shifts         = []          
shifts_per_mol     = []        
bond_type_counts   = Counter()   
solvent_counts     = Counter()   
ring_size_presence = Counter()   
frac_aromatic      = []          

for mol in supplier:
    n_total += 1
    if mol is None:
        continue
    n_parsed += 1

    # atom presence (binary per molecule)
    symbols_in_mol = set(a.GetSymbol() for a in mol.GetAtoms())
    for sym in symbols_in_mol:
        atom_presence[sym] += 1

    # molecule size & weight
    heavy_atom_counts.append(mol.GetNumHeavyAtoms())
    mol_weights.append(Descriptors.MolWt(mol))

    # bond types 
    for bond in mol.GetBonds():
        bond_type_counts[str(bond.GetBondTypeAsDouble())] += 1

    # ring sizes 
    ri = mol.GetRingInfo()
    sizes_in_mol = set(len(r) for r in ri.AtomRings())
    for s in sizes_in_mol:
        ring_size_presence[s] += 1

    # aromaticity 
    n_arom = sum(1 for a in mol.GetAtoms() if a.GetIsAromatic())
    frac_aromatic.append(n_arom / mol.GetNumHeavyAtoms())

    # solvent 
    solvent_raw = mol.GetPropsAsDict().get("Solvent", "")
    if solvent_raw:
        first = re.split(r"\s+\d+:", solvent_raw)[0]
        solvent = re.sub(r"^\d+:", "", first).strip()
        solvent_counts[solvent] += 1

    # 13C shifts 
    prop = mol.GetPropsAsDict().get("Spectrum 13C 0", "")
    if not prop:
        continue
    shifts = []
    for entry in prop.split("|"):
        entry = entry.strip()
        if not entry:
            continue
        parts = entry.split(";")
        if len(parts) < 3:
            continue
        try:
            v = float(parts[0])
        except ValueError:
            continue
        if SHIFT_MIN <= v <= SHIFT_MAX:
            shifts.append(v)
    if shifts:
        n_with_13c += 1
        all_shifts.extend(shifts)
        shifts_per_mol.append(len(shifts))

print(f"Total records : {n_total:,}")
print(f"Parsed OK     : {n_parsed:,}")
print(f"With 13C data : {n_with_13c:,}")
print(f"Total 13C shifts collected: {len(all_shifts):,}")

## Atom frequency (% of molecules containing each element)

In [ ]:
# Sort by frequency descending, show top-20
top_atoms = atom_presence.most_common(20)
syms  = [s for s, _ in top_atoms]
freqs = [100 * c / n_parsed for _, c in top_atoms]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(syms, freqs, color="steelblue", edgecolor="white")
ax.set_ylabel("% of molecules containing element")
ax.set_title("Atom presence frequency (binary per molecule)")
ax.set_ylim(0, 105)
for bar, v in zip(bars, freqs):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 1, f"{v:.1f}%",
            ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

## 13C chemical shift distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_shifts, bins=100, color="darkorange", edgecolor="white")
ax.set_xlabel("13C chemical shift (ppm)")
ax.set_ylabel("# atoms")
ax.set_title(f"13C shift distribution ({len(all_shifts):,} labelled atoms)")
for p in [25, 50, 75]:
    v = np.percentile(all_shifts, p)
    ax.axvline(v, linestyle="--", linewidth=0.9, label=f"p{p}={v:.1f}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Bond type distribution

In [ ]:
label_map = {"1.0": "Single", "1.5": "Aromatic", "2.0": "Double", "3.0": "Triple"}
bond_labels = [label_map.get(k, k) for k in bond_type_counts]
bond_vals   = list(bond_type_counts.values())

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(bond_labels, bond_vals, color="cornflowerblue", edgecolor="white")
ax.set_ylabel("Total bond count")
ax.set_title("Bond type distribution across all molecules")
for i, v in enumerate(bond_vals):
    ax.text(i, v + max(bond_vals) * 0.01, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()